In [ ]:
import warnings
warnings.filterwarnings('ignore')

#!pip install pgmpy
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

In [ ]:
# Load your dataset
#df = pd.read_csv("Put Your Dataset Name Here")
df = pd.read_csv("c:\\risk\\lendingclub_sample_25000.csv")
df.head()

In [ ]:
# Select relevant columns and drop missing
df = df[['int_rate', 'dti', 'fico_range_high', 'bad_loan']].dropna()
df['bad_loan'] = pd.to_numeric(df['bad_loan'], errors='coerce')
df = df.dropna()

In [ ]:
# Discretize features into 3 bins (low, medium, high)
bin_features = ['int_rate', 'dti', 'fico_range_high']
discretizer = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
df[bin_features] = discretizer.fit_transform(df[bin_features])
df[bin_features] = df[bin_features].astype(int)
df['bad_loan'] = df['bad_loan'].astype(int)

In [ ]:
# Define Bayesian network structure
from pgmpy.models import DiscreteBayesianNetwork  # Changed from BayesianNetwork to DiscreteBayesianNetwork

model = DiscreteBayesianNetwork()  # Using the recommended class
model.add_nodes_from(['int_rate', 'dti', 'fico_range_high', 'bad_loan'])
model.add_edges_from([
    ('int_rate', 'bad_loan'),
    ('dti', 'bad_loan'),
    ('fico_range_high', 'bad_loan')
])

# Fit model using Bayesian Estimator
model.fit(df, estimator=BayesianEstimator, prior_type='BDeu')

# Perform inference
inference = VariableElimination(model)

In [ ]:
# Query: What's the probability of bad_loan given suspicious pattern?
query = inference.query(
    variables=['bad_loan'],
    evidence={'int_rate': 2, 'dti': 2, 'fico_range_high': 0}
)

In [ ]:
df['bad_loan'].mean()

In [ ]:
print(query)

In [ ]:
#Loop through all possibilities

In [ ]:
import itertools
rows = []
for i, j, k in itertools.product(range(3), repeat=3):
    q = inference.query(
        variables=['bad_loan'],
        evidence={'int_rate': i, 'dti': j, 'fico_range_high': k}
    )
    prob = q.values[1]
    rows.append({'int_rate': i, 'dti': j, 'fico_range_high': k, 'P(bad_loan)': prob})

In [ ]:
#df_results = pd.DataFrame(rows)
#df_results = df_results.round(3)  # Round all numeric columns to 3 decimals
#df_results.to_csv("c:\\risk\\bayesian_fraud_risk_grid.csv", index=False)

df_results = pd.DataFrame(rows)

In [ ]:
# Map numeric bins to labels
label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
df_results['int_rate'] = df_results['int_rate'].map(label_map)
df_results['dti'] = df_results['dti'].map(label_map)
df_results['fico_range_high'] = df_results['fico_range_high'].map(label_map)

In [ ]:
# Round probability values and export
#df_results.to_csv("Put Your Output Dataset Name Here", index=False)
df_results = df_results.round(3)
df_results.to_csv("c:\\risk\\bayesian_bad_loan_risk.csv", index=False)

In [ ]:
# Fit the model
model.fit(df, estimator=BayesianEstimator, prior_type="BDeu")

In [ ]:
# Print CPDs
for cpd in model.get_cpds():
    print("CPD of", cpd.variable)
    print(cpd)

In [ ]:
print("Overall default rate:", df['bad_loan'].mean())
print()
print(df_results.sort_values('P(bad_loan)', ascending=False).to_string(index=False))